In [2]:
import os
import pytest
import responses
from openai import OpenAI
from pydantic import BaseModel, Field

# 🔑 Set your API key if it's not already in your system environment variables

os.environ["OPENAI_API_KEY"] = "<Enter your APIs>"

# 1. Define our Pydantic tool blueprint for a banking system
class FundTransferSchema(BaseModel):
    recipient_account_id: str = Field(description="The alphanumeric target account string.")
    amount_usd: float = Field(description="The exact amount of money to transfer as a decimal float.")

# 2. This is our application's "Bridge" function.
def call_backend_banking_api(parsed_tool_call: FundTransferSchema):
    import requests
    
    backend_url = "https://api.securebank.internal/v1/transfers"
    
    payload = {
        "target": parsed_tool_call.recipient_account_id,
        "amount": parsed_tool_call.amount_usd
    }
    
    response = requests.post(backend_url, json=payload, timeout=5)
    return response

print("✅ Cell 1 Executed: Environment set up, schema defined, and banking bridge function initialized.")

✅ Cell 1 Executed: Environment set up, schema defined, and banking bridge function initialized.


In [3]:
@responses.activate
def test_mock_backend_api_bridge_success():
    """
    Test Case 1: Verify our application can parse LLM-style data 
    and hit the mock API, handling a successful '200 OK' response.
    """
    # Arrange: Intercept any outgoing network call to our bank URL and return a fake success block
    responses.add(
        responses.POST,
        "https://api.securebank.internal/v1/transfers",
        json={"status": "TXN_SUCCESS", "reference_id": "99823"},
        status=200
    )
    
    # Act: Simulate a successful tool generation from an AI model
    simulated_llm_output = FundTransferSchema(recipient_account_id="ACC-7711", amount_usd=250.50)
    api_response = call_backend_banking_api(simulated_llm_output)
    
    # Assert: Prove that the bridge formatted the payload and handled the api output correctly
    assert api_response.status_code == 200
    assert api_response.json()["status"] == "TXN_SUCCESS"
    print("\n[PASSED] Mock Test: Successfully simulated a clean 200 OK API transaction bridge.")

# This line forces Jupyter to execute the specific test function written above right inside the cell!
pytest.main(["-v", "-s", "-k", "test_mock_backend_api_bridge_success"])

============================= test session starts ==============================
platform darwin -- Python 3.13.9, pytest-8.4.2, pluggy-1.5.0 -- /Users/amritansh/anaconda3/bin/python
cachedir: .pytest_cache
rootdir: /Users/amritansh/Documents/EY_AI_Test/D3_EY
plugins: anyio-4.13.0
collecting ... collected 0 items

=============================== warnings summary ===============================
../../../anaconda3/lib/python3.13/site-packages/_pytest/config/__init__.py:1290
  /Users/amritansh/anaconda3/lib/python3.13/site-packages/_pytest/config/__init__.py:1290: PytestAssertRewriteWarning: Module already imported so cannot be rewritten; anyio
    self._mark_plugins_for_rewrite(hook, disable_autoload)

-- Docs: https://docs.pytest.org/en/stable/how-to/capture-warnings.html
============================== 1 warning in 0.00s ==============================


<ExitCode.NO_TESTS_COLLECTED: 5>

In [4]:
@responses.activate
def test_live_llm_to_mock_api_pipeline():
    """
    Test Case 2: End-to-End integration test. Feed raw human text to OpenAI,
    let it generate the tool arguments via Pydantic, send it across our bridge,
    and assert how the system handles a backend failure ('400 Bad Request').
    """
    # Arrange: Configure our mock server to reject the transaction if the AI triggers it
    responses.add(
        responses.POST,
        "https://api.securebank.internal/v1/transfers",
        json={"error": "INSUFFICIENT_FUNDS"},
        status=400
    )
    
    client = OpenAI()
    user_prompt = "Can you send two hundred and fifty dollars to account number ACC-7711 right now?"
    
    # Act Step A: Feed chaotic human text to the AI and extract structured tool arguments
    completion = client.beta.chat.completions.parse(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": "Extract banking tool parameters accurately."},
            {"role": "user", "content": user_prompt}
        ],
        response_format=FundTransferSchema
    )
    parsed_ai_tool_call = completion.choices[0].message.parsed
    
    # Act Step B: Pass that live AI data straight into our backend API bridge code
    api_response = call_backend_banking_api(parsed_ai_tool_call)
    
    # Assert: Verify the data was extracted perfectly by the LLM AND caught cleanly by our API error framework
    assert parsed_ai_tool_call.recipient_account_id == "ACC-7711"
    assert parsed_ai_tool_call.amount_usd == 250.00
    assert api_response.status_code == 400
    assert api_response.json()["error"] == "INSUFFICIENT_FUNDS"
    print("\n[PASSED] Integration Test: End-to-end pipeline verified. The LLM parsed accurately, passed it to our bridge, and our system isolated a 400 error cleanly.")

# Execute this integration test function inside this notebook cell
pytest.main(["-v", "-s", "-k", "test_live_llm_to_mock_api_pipeline"])

============================= test session starts ==============================
platform darwin -- Python 3.13.9, pytest-8.4.2, pluggy-1.5.0 -- /Users/amritansh/anaconda3/bin/python
cachedir: .pytest_cache
rootdir: /Users/amritansh/Documents/EY_AI_Test/D3_EY
plugins: anyio-4.13.0
collecting ... collected 0 items

=============================== warnings summary ===============================
../../../anaconda3/lib/python3.13/site-packages/_pytest/config/__init__.py:1290
  /Users/amritansh/anaconda3/lib/python3.13/site-packages/_pytest/config/__init__.py:1290: PytestAssertRewriteWarning: Module already imported so cannot be rewritten; anyio
    self._mark_plugins_for_rewrite(hook, disable_autoload)

-- Docs: https://docs.pytest.org/en/stable/how-to/capture-warnings.html
============================== 1 warning in 0.01s ==============================


<ExitCode.NO_TESTS_COLLECTED: 5>

In [5]:
# Run ALL tests simultaneously to view the full pipeline health report
print("📋 Running Full System Verification Test Suite...")
pytest.main(["-v", "-s"])

📋 Running Full System Verification Test Suite...
============================= test session starts ==============================
platform darwin -- Python 3.13.9, pytest-8.4.2, pluggy-1.5.0 -- /Users/amritansh/anaconda3/bin/python
cachedir: .pytest_cache
rootdir: /Users/amritansh/Documents/EY_AI_Test/D3_EY
plugins: anyio-4.13.0
collecting ... collected 0 items

=============================== warnings summary ===============================
../../../anaconda3/lib/python3.13/site-packages/_pytest/config/__init__.py:1290
  /Users/amritansh/anaconda3/lib/python3.13/site-packages/_pytest/config/__init__.py:1290: PytestAssertRewriteWarning: Module already imported so cannot be rewritten; anyio
    self._mark_plugins_for_rewrite(hook, disable_autoload)

-- Docs: https://docs.pytest.org/en/stable/how-to/capture-warnings.html
============================== 1 warning in 0.00s ==============================


<ExitCode.NO_TESTS_COLLECTED: 5>